# Genesis — IE 643 | Day 2: Method B + evaluation set (both T4s in parallel)
**Settings:** Accelerator **GPU T4 x2** · Internet **On** · Persistence **Files only**

**How to run:** click **Save Version → Save & Run All (Commit)**. It runs top to bottom in the background
(you can close the browser), about **3 hours**. Results appear in the version's **Output** tab.

What it produces — 4 prompt sequences × 5 clips (~10 s each), each with:

| Method | How clip k+1 is made |
|---|---|
| `baseline` | independent text-to-video clip, hard cut |
| `A_lastframe` (m = 1) | conditioned on the **last frame** of clip k |
| `B_m2` (m = 2) | conditioned on the **last 5 frames** of clip k → 2 clean latent frames |
| `B_m3` (m = 3, fox only, ablation) | conditioned on the **last 9 frames** → 3 clean latent frames |

Clip 0 is shared by all methods, and every clip index uses the same prompt and seed across methods, so the
chaining strategy is the only thing that changes.

### Method B in one equation (flow matching, per-token timesteps)
TI2V-5B gives each latent token its own flow-matching time *s* (1 = noise, 0 = clean).
For the new clip's latent **z** with *T′* = 13 latent frames, Method B sets

$$z^{(k+1)}_{0:m} \leftarrow \mathrm{norm}\big(\mathcal{E}(\text{last } 1+4(m-1) \text{ frames of clip } k)\big), \qquad s_{0:m} = 0,$$

re-injects those *m* clean latent frames after every solver step, and denoises only frames *m … T′−1*.
Method A is the special case *m* = 1 (exactly the model's built-in image-to-video conditioning).
The first 1+4(m−1) frames of each chained clip reproduce the context and are dropped when stitching.

## 1 — Install (torchao removed: Kaggle's copy breaks current diffusers)

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg huggingface_hub
import diffusers, transformers, torch
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__,
      "| GPUs", torch.cuda.device_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 — Configuration: prompts, seeds and the per-GPU plan

In [ ]:
import os, json, time, gc, shutil, re, html, sys, subprocess
import numpy as np
import torch

os.environ["HF_HOME"] = "/tmp/hf_home"
MODEL_ID    = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
WEIGHTS_DIR = "/tmp/wan22_ti2v5b"
OUT         = "/kaggle/working/genesis"
for d in ["embeds", "videos", "latents", "frames", "logs", "manifests", "code", "figures"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

HEIGHT, WIDTH, NUM_FRAMES, FPS = 480, 832, 49, 24
STEPS, GUIDANCE, FLOW_SHIFT, BASE_SEED = 30, 5.0, 3.0, 1000
N_CLIPS = 5

NEG_PROMPT = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, "
              "overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, "
              "poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, "
              "messy background, three legs, many people in the background, walking backwards")

PROMPT_SEQS = {
    "fox": [
        "A red fox with a bushy white-tipped tail walks slowly through a snowy pine forest at sunrise, soft golden light, cinematic, steady tracking shot.",
        "The same red fox reaches the edge of the snowy pine forest and stops beside a frozen stream, sniffing the snow, soft golden sunrise light, steady tracking shot.",
        "The same red fox jumps across the frozen stream, snow spraying from its paws, soft golden sunrise light, steady tracking shot.",
        "The same red fox trots across an open snowy meadow beyond the stream, its breath visible in the cold air, soft golden sunrise light, steady tracking shot.",
        "The same red fox curls up to rest at the foot of a large pine tree on the snowy meadow, soft golden sunrise light, slow push-in shot.",
    ],
    "boat": [
        "A small white paper boat floats along a rain-filled street gutter at dusk, neon shop signs reflecting in the water, close-up, cinematic.",
        "The same white paper boat spins gently as it drifts past a storm drain, neon reflections rippling on the water, close-up, cinematic.",
        "The same white paper boat drifts out of the gutter into a wide puddle under a streetlight as rain keeps falling, neon reflections, close-up, cinematic.",
        "Raindrops patter around the same white paper boat as it floats slowly across the wide puddle, neon reflections shimmering, close-up, cinematic.",
        "The same white paper boat, now soggy, comes to rest against the curb as the rain stops, calm neon reflections on the water, close-up, cinematic.",
    ],
    "balloon": [
        "A red and yellow striped hot air balloon rises slowly above a misty green valley at dawn, soft pastel sky, wide aerial shot.",
        "The same red and yellow striped hot air balloon drifts over a winding river in the misty valley at dawn, soft pastel sky, wide aerial shot.",
        "The same red and yellow striped hot air balloon floats above a small village with red rooftops in the valley, morning light breaking through the mist, wide aerial shot.",
        "The same red and yellow striped hot air balloon passes over rolling green hills dotted with trees, its burner flaring orange, warm morning light, wide aerial shot.",
        "The same red and yellow striped hot air balloon glides toward distant snow-capped mountains under a clear blue morning sky, wide aerial shot.",
    ],
    "car": [
        "A vintage red convertible car drives along a winding coastal cliff road at sunset, the ocean sparkling below, cinematic drone tracking shot.",
        "The same vintage red convertible car rounds a bend on the coastal cliff road as a white lighthouse comes into view, golden sunset light, cinematic drone tracking shot.",
        "The same vintage red convertible car drives past the white lighthouse on the coastal road, waves crashing on the rocks below, golden sunset light, cinematic drone tracking shot.",
        "The same vintage red convertible car descends the coastal road into a small seaside town with white houses, warm sunset light, cinematic drone tracking shot.",
        "The same vintage red convertible car parks at a seaside viewpoint as the sun sets over the ocean, orange sky, cinematic drone shot slowly circling.",
    ],
}
SEEDS = {s: [BASE_SEED + 100 * j + i for i in range(N_CLIPS)] for j, s in enumerate(PROMPT_SEQS)}
METHODS = [["baseline", 1], ["A_lastframe", 1], ["B_m2", 2]]
EXTRA = {"fox": [["B_m3", 3]]}            # context-length ablation on one sequence

def job(seq):
    return {"seq": seq, "n_clips": N_CLIPS, "seeds": SEEDS[seq], "methods": METHODS + EXTRA.get(seq, [])}

N_GPU = torch.cuda.device_count()
if N_GPU >= 2:
    WORKERS = {"0": [job("fox"), job("boat")], "1": [job("balloon"), job("car")]}
else:
    WORKERS = {"0": [job(s) for s in PROMPT_SEQS]}

CONFIG = {"OUT": OUT, "WEIGHTS_DIR": WEIGHTS_DIR, "EMBEDS": f"{OUT}/embeds/day2_embeds.pt",
          "HEIGHT": HEIGHT, "WIDTH": WIDTH, "NUM_FRAMES": NUM_FRAMES, "STEPS": STEPS, "GUIDANCE": GUIDANCE,
          "FLOW_SHIFT": FLOW_SHIFT, "FPS": FPS, "DEVICE": "cuda", "VAE_OFFLOAD": True, "DTYPE": "float16"}
PLAN = f"{OUT}/logs/day2_plan.json"
json.dump({"config": CONFIG, "workers": WORKERS, "prompts": PROMPT_SEQS, "negative_prompt": NEG_PROMPT}, open(PLAN, "w"), indent=2)
n_total = sum(1 + (N_CLIPS - 1) * len(j["methods"]) for w in WORKERS.values() for j in w)
print(f"{n_total} clips planned on {len(WORKERS)} worker(s):",
      {w: sum(1 + (N_CLIPS - 1) * len(j["methods"]) for j in js) for w, js in WORKERS.items()})

## 3 — Encode every prompt once with UMT5-XXL, then delete the text encoder

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, UMT5EncoderModel
import ftfy

if not os.path.exists(CONFIG["EMBEDS"]):
    snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*"])
    tok = AutoTokenizer.from_pretrained(f"{WEIGHTS_DIR}/tokenizer")
    if N_GPU >= 2:
        te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.float32,
                                              device_map="auto", max_memory={0: "14GiB", 1: "14GiB", "cpu": "20GiB"})
        te_device = "cuda:0"
    else:
        te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.bfloat16)
        te_device = "cpu"
    te.eval()

    def prompt_clean(t):
        t = ftfy.fix_text(t)
        t = html.unescape(html.unescape(t)).strip()
        return re.sub(r"\s+", " ", t).strip()

    @torch.no_grad()
    def t5_embed(text, max_len=512):
        enc = tok([prompt_clean(text)], padding="max_length", max_length=max_len, truncation=True,
                  add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
        ids, mask = enc.input_ids.to(te_device), enc.attention_mask.to(te_device)
        n = int(mask.sum())
        h = te(ids, mask).last_hidden_state[0, :n].float().cpu()
        out = torch.zeros(max_len, h.shape[-1]); out[:n] = h
        return out.unsqueeze(0)

    t0 = time.time()
    E = {"__negative__": t5_embed(NEG_PROMPT)}
    for s, ps in PROMPT_SEQS.items():
        for i, p in enumerate(ps):
            E[f"{s}_{i}"] = t5_embed(p)
    torch.save(E, CONFIG["EMBEDS"])
    print(f"Encoded {len(E)} prompts in {time.time() - t0:.0f}s")
    del te; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"{WEIGHTS_DIR}/text_encoder", ignore_errors=True)
    shutil.rmtree("/tmp/hf_home", ignore_errors=True)
else:
    print("Embeddings already exist:", CONFIG["EMBEDS"])

## 4 — Download transformer, VAE and scheduler

In [ ]:
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "transformer/*", "vae/*", "scheduler/*"])
!du -sh {WEIGHTS_DIR}/* | cat

## 5 — Project code (saved into the output so it is part of the submission)

In [ ]:
%%writefile /kaggle/working/genesis/code/genesis_lib.py
"""
genesis_lib.py — Genesis (IE 643): long-form video by chaining short Wan2.2 TI2V-5B clips.

Chaining methods
  baseline : every clip generated independently from text (T2V) and joined with hard cuts.
  A (m=1)  : clip k+1 is conditioned on the LAST FRAME of clip k (the model's own TI2V first-frame conditioning).
  B (m>=2) : clip k+1 is conditioned on the last K = 1 + 4(m-1) FRAMES of clip k. They are re-encoded by the
             causal 3D VAE into m clean latent frames, placed at the start of the new clip's latent; those tokens
             get flow-matching timestep 0 (clean) and are re-injected after every solver step, so the model
             denoises the rest of the clip around a context that shows the motion, not just a still frame.

Flow matching (what the model was trained with)
  x_s = (1 - s) * x_0 + s * eps,  s in [0, 1]   (s = 1 pure noise, s = 0 clean)
  model predicts velocity v = eps - x_0;  loss = || (eps - x_0) - v_theta(x_s, s, text) ||^2
  TI2V-5B gives every latent token its own timestep: conditioning tokens get s = 0, the rest the current s.
"""
import json, os, time

import numpy as np
import torch
import diffusers
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, WanPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from diffusers.utils.torch_utils import get_module_execution_device, randn_tensor
from diffusers.pipelines.wan.pipeline_wan_i2v import retrieve_latents
from PIL import Image

TEMPORAL = 4      # Wan2.2 VAE temporal compression
TAIL_KEEP = 9     # pixel frames kept losslessly per clip = context for m <= 3


def context_frames(m: int) -> int:
    """Pixel frames the VAE maps to m latent frames (first frame alone, then groups of 4)."""
    return 1 + TEMPORAL * (m - 1)


def new_frames_per_clip(num_frames: int, m: int) -> int:
    """Frames a chained clip adds to the video (its first K frames repeat the context and are dropped)."""
    return num_frames - context_frames(m)


def clips_for_duration(seconds: float, fps: int, num_frames: int, m: int) -> int:
    """Number of clips needed so the stitched video lasts at least `seconds`."""
    need = max(int(round(seconds * fps)), num_frames)
    return 1 + int(np.ceil((need - num_frames) / new_frames_per_clip(num_frames, m)))


def _prepare_latents_ctx(self, image, batch_size, num_channels_latents=48, height=480, width=832,
                         num_frames=49, dtype=None, device=None, generator=None, latents=None, last_image=None):
    """Replacement for WanImageToVideoPipeline.prepare_latents (expand_timesteps / TI2V path).

    If self._genesis_ctx (uint8 array, K x H x W x 3) is set, it is encoded into m = (K-1)/4 + 1 clean latent
    frames and the first m latent frames are masked as 'known' (timestep 0, re-injected every step).
    Otherwise falls back to the stock single-image behaviour.
    """
    ctx = getattr(self, "_genesis_ctx", None)
    if ctx is None:
        return WanImageToVideoPipeline.prepare_latents(self, image, batch_size, num_channels_latents, height, width,
                                                       num_frames, dtype, device, generator, latents, last_image)
    K = ctx.shape[0]
    assert (K - 1) % TEMPORAL == 0, f"context length {K} must be 1 + 4(m-1)"
    assert ctx.shape[1:3] == (height, width), f"context frames {ctx.shape[1:3]} != ({height}, {width})"

    T = (num_frames - 1) // self.vae_scale_factor_temporal + 1
    h, w = height // self.vae_scale_factor_spatial, width // self.vae_scale_factor_spatial
    shape = (batch_size, num_channels_latents, T, h, w)
    if latents is None:
        latents = randn_tensor(shape, generator=generator, device=device, dtype=dtype)
    else:
        latents = latents.to(device=device, dtype=dtype)

    vid = torch.from_numpy(np.ascontiguousarray(ctx)).float().div(127.5).sub(1.0)   # K,H,W,3 in [-1, 1]
    vid = vid.permute(3, 0, 1, 2).unsqueeze(0)                                     # 1,3,K,H,W
    vae_dev = get_module_execution_device(self.vae)
    with torch.no_grad():
        lc = retrieve_latents(self.vae.encode(vid.to(vae_dev, self.vae.dtype)), sample_mode="argmax")
    lc = lc.to(device=device, dtype=dtype)
    mean = torch.tensor(self.vae.config.latents_mean).view(1, -1, 1, 1, 1).to(device, dtype)
    inv_std = 1.0 / torch.tensor(self.vae.config.latents_std).view(1, -1, 1, 1, 1).to(device, dtype)
    lc = (lc - mean) * inv_std

    m = lc.shape[2]
    assert m < T, "context must be shorter than the clip"
    cond = torch.zeros_like(latents)
    cond[:, :, :m] = lc.expand(batch_size, -1, -1, -1, -1)
    mask = torch.ones(1, 1, T, h, w, dtype=dtype, device=device)
    mask[:, :, :m] = 0
    if getattr(self, "_genesis_vae_offload", False):
        self.vae.to("cpu")
        torch.cuda.empty_cache()
    return latents, cond, mask


class _T2V(WanPipeline):
    """Stock T2V pipeline, but always computes on the GPU even while the VAE sits on the CPU."""
    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class _CtxI2V(WanImageToVideoPipeline):
    """TI2V pipeline whose conditioning can be a multi-frame context (see _prepare_latents_ctx)."""
    _genesis_ctx = None
    _genesis_vae_offload = False
    prepare_latents = _prepare_latents_ctx

    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class Genesis:
    """Holds one TI2V-5B transformer + VAE on one GPU and generates / chains clips."""

    def __init__(self, weights_dir, height=480, width=832, num_frames=49, steps=30, guidance=5.0,
                 flow_shift=3.0, fps=24, device="cuda", vae_offload=True, dtype=torch.float16):
        self.h, self.w, self.nf, self.steps, self.cfg, self.fps = height, width, num_frames, steps, guidance, fps
        self.device, self.vae_offload, self.dtype = device, vae_offload, dtype
        idx = json.load(open(f"{weights_dir}/model_index.json"))
        expand = bool(idx.get("expand_timesteps", False))
        assert expand, "TI2V-5B must load with expand_timesteps=True (per-token timesteps)"

        self.vae = AutoencoderKLWan.from_pretrained(f"{weights_dir}/vae", torch_dtype=torch.float32)
        self.vae.enable_tiling()
        self.vae.to("cpu" if vae_offload else device)
        try:
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype, device_map=device)
        except Exception as e:  # older/newer API without device_map support for this model
            print("device_map load failed, loading on CPU first:", repr(e)[:200])
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype).to(device)
        self.transformer.eval()

        cfg = json.load(open(f"{weights_dir}/scheduler/scheduler_config.json"))
        cls = getattr(diffusers, cfg["_class_name"])
        sched = cls.from_pretrained(f"{weights_dir}/scheduler")
        key = "flow_shift" if "flow_shift" in sched.config else ("shift" if "shift" in sched.config else None)
        if key:
            sched = cls.from_config(sched.config, **{key: flow_shift})

        common = dict(tokenizer=None, text_encoder=None, vae=self.vae, scheduler=sched,
                      transformer=self.transformer, expand_timesteps=expand)
        self.t2v = _T2V(**common)
        self.i2v = _CtxI2V(**common)
        self.i2v._genesis_vae_offload = vae_offload
        for p in (self.t2v, self.i2v):
            p._genesis_device = device
            p.set_progress_bar_config(disable=True)

    # ---------------------------------------------------------------- helpers
    def _emb(self, E, key):
        return E[key].to(self.device, self.dtype)

    def _kwargs(self, E, seed):
        return dict(negative_prompt_embeds=self._emb(E, "__negative__"), height=self.h, width=self.w,
                    num_frames=self.nf, num_inference_steps=self.steps, guidance_scale=self.cfg,
                    generator=torch.Generator("cpu").manual_seed(int(seed)), output_type="latent")

    @torch.no_grad()
    def _decode(self, lat):
        v = self.vae
        v.to(self.device)
        try:
            z = lat.to(self.device, v.dtype)
            mean = torch.tensor(v.config.latents_mean).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            std = torch.tensor(v.config.latents_std).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            video = v.decode(z * std + mean, return_dict=False)[0]
            return self.t2v.video_processor.postprocess_video(video, output_type="np")[0]  # F,H,W,3 in [0,1]
        finally:
            if self.vae_offload:
                v.to("cpu")
            torch.cuda.empty_cache()

    def decode(self, lat):
        try:
            return self._decode(lat)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self.transformer.to("cpu")
            torch.cuda.empty_cache()
            try:
                return self._decode(lat)
            finally:
                self.transformer.to(self.device)

    # ---------------------------------------------------------------- generation
    def text_clip(self, E, key, seed):
        """Independent clip from text only (baseline, and clip 0 of every chain)."""
        lat = self.t2v(prompt_embeds=self._emb(E, key), **self._kwargs(E, seed)).frames
        return self.decode(lat), lat

    def context_clip(self, ctx_uint8, E, key, seed):
        """Clip conditioned on K = 1 + 4(m-1) previous frames (m = 1 is Method A, m >= 2 is Method B)."""
        if self.vae_offload:
            self.vae.to(self.device)
        self.i2v._genesis_ctx = ctx_uint8
        try:
            dummy = Image.fromarray(ctx_uint8[-1])   # required argument; real conditioning is _genesis_ctx
            lat = self.i2v(image=dummy, prompt_embeds=self._emb(E, key), **self._kwargs(E, seed)).frames
        finally:
            self.i2v._genesis_ctx = None
            if self.vae_offload:
                self.vae.to("cpu")
        return self.decode(lat), lat


# -------------------------------------------------------------------- storage
def to_uint8(frames):
    return (np.clip(frames, 0.0, 1.0) * 255.0).round().astype(np.uint8)


def frames_ok(frames):
    return bool(np.isfinite(frames).all() and frames.max() > 0.02)


def save_clip(out, tag, frames, lat, fps):
    export_to_video(list(frames), f"{out}/videos/{tag}.mp4", fps=fps)
    torch.save(lat.detach().cpu().half(), f"{out}/latents/{tag}.pt")
    np.save(f"{out}/frames/{tag}_tail.npy", to_uint8(frames[-TAIL_KEEP:]))


def clip_done(out, tag):
    return os.path.exists(f"{out}/frames/{tag}_tail.npy") and os.path.exists(f"{out}/videos/{tag}.mp4")


def load_tail(out, tag):
    return np.load(f"{out}/frames/{tag}_tail.npy")


def read_video(path):
    import imageio.v3 as iio
    return np.stack(list(iio.imiter(path))).astype(np.float32) / 255.0


def log_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


# -------------------------------------------------------------------- one sequence, one method
def run_sequence(G, E, out, seq, n_clips, seeds, method, m, log_path):
    """method: 'baseline' or a chain name; m: context latent frames (1 = Method A)."""
    clips = []   # (frames float, kept-from index)
    for i in range(n_clips):
        if method == "baseline" or i == 0:
            tag = f"{seq}_baseline_c{i}"
        else:
            tag = f"{seq}_{method}_c{i}"
        if clip_done(out, tag):
            frames = read_video(f"{out}/videos/{tag}.mp4")
            stats = {"resumed": True}
        else:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            if method == "baseline" or i == 0:
                frames, lat = G.text_clip(E, f"{seq}_{i}", seeds[i])
            else:
                K = context_frames(m)
                ctx = load_tail(out, prev_tag)[-K:]
                frames, lat = G.context_clip(ctx, E, f"{seq}_{i}", seeds[i])
            stats = {"sec": round(time.time() - t0, 1),
                     "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0,
                     "frames_ok": frames_ok(frames)}
            save_clip(out, tag, frames, lat, G.fps)
        drop = 0 if (method == "baseline" or i == 0) else context_frames(m)
        clips.append(frames[drop:])
        prev_tag = tag
        rec = {"seq": seq, "method": method, "m": m, "clip": i, "tag": tag, **stats}
        log_jsonl(log_path, rec)
        print(rec, flush=True)

    full = np.concatenate(clips, axis=0)
    export_to_video(list(full), f"{out}/videos/{seq}_{method}_FULL.mp4", fps=G.fps)
    lengths = [int(c.shape[0]) for c in clips]
    json.dump({"seq": seq, "method": method, "m": m, "clip_lengths": lengths,
               "boundaries": [int(b) for b in np.cumsum(lengths)[:-1]], "seconds": round(full.shape[0] / G.fps, 2)},
              open(f"{out}/manifests/{seq}_{method}.json", "w"), indent=2)
    return full

In [ ]:
%%writefile /kaggle/working/genesis/code/genesis_worker.py
"""genesis_worker.py — runs one GPU's share of the Day 2 plan (launched by the notebook, one process per GPU)."""
import argparse, json, os, sys
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import genesis_lib as gl

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
ap.add_argument("--worker", required=True)
a = ap.parse_args()

plan = json.load(open(a.plan))
C = plan["config"]
OUT = C["OUT"]
G = gl.Genesis(C["WEIGHTS_DIR"], C["HEIGHT"], C["WIDTH"], C["NUM_FRAMES"], C["STEPS"], C["GUIDANCE"],
               C["FLOW_SHIFT"], C["FPS"], device=C.get("DEVICE", "cuda"), vae_offload=C.get("VAE_OFFLOAD", True),
               dtype=getattr(torch, C.get("DTYPE", "float16")))
open(f"{OUT}/logs/worker{a.worker}.ready", "w").write("ok")
print(f"worker {a.worker} loaded", flush=True)

E = torch.load(C["EMBEDS"])
log = f"{OUT}/logs/runs_worker{a.worker}.jsonl"
for job in plan["workers"][a.worker]:
    for method, m in job["methods"]:
        gl.run_sequence(G, E, OUT, job["seq"], job["n_clips"], job["seeds"], method, m, log)
open(f"{OUT}/logs/worker{a.worker}.done", "w").write("ok")
print(f"worker {a.worker} done", flush=True)

## 6 — Launch one worker per GPU and wait (~2.5 h)
Workers start one after another (the second waits until the first has loaded its weights, to limit RAM).
Every finished clip is saved immediately; if the run stops, re-running skips finished clips.

In [ ]:
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
procs = {}
for w in sorted(WORKERS):
    e = dict(env, CUDA_VISIBLE_DEVICES=str(int(w) % max(N_GPU, 1)))
    logf = open(f"{OUT}/logs/worker{w}.log", "a")
    procs[w] = subprocess.Popen([sys.executable, f"{OUT}/code/genesis_worker.py", "--plan", PLAN, "--worker", w],
                                env=e, stdout=logf, stderr=subprocess.STDOUT)
    while not os.path.exists(f"{OUT}/logs/worker{w}.ready") and procs[w].poll() is None:
        time.sleep(15)
    print(f"worker {w} started (pid {procs[w].pid})", "— loaded" if procs[w].poll() is None else "— EXITED early")

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for w in procs:
        lines = open(f"{OUT}/logs/worker{w}.log").read().strip().splitlines()
        print(f"[{(time.time() - t0) / 60:5.0f} min] worker {w}: {lines[-1][:160] if lines else '(no output yet)'}")
for w, p in procs.items():
    print(f"worker {w} exit code {p.returncode}")
    if p.returncode != 0:
        print(open(f"{OUT}/logs/worker{w}.log").read()[-6000:])

## 7 — Qualitative outputs: seam grids, side-by-side comparison videos, timing summary

In [ ]:
sys.path.insert(0, f"{OUT}/code")
import genesis_lib as gl
from PIL import Image, ImageDraw
from diffusers.utils import export_to_video

def label(img, text):
    im = Image.fromarray(img); d = ImageDraw.Draw(im)
    d.rectangle([0, 0, 8 + 7 * len(text), 18], fill=(0, 0, 0)); d.text((4, 3), text, fill=(255, 255, 255))
    return np.asarray(im)

def small(f, s=2):
    return np.asarray(Image.fromarray(gl.to_uint8(f)).resize((WIDTH // s, HEIGHT // s)))

for seq in PROMPT_SEQS:
    methods = [m for m, _ in METHODS + EXTRA.get(seq, [])]
    vids, rows = {}, []
    for meth in methods:
        man_p, vid_p = f"{OUT}/manifests/{seq}_{meth}.json", f"{OUT}/videos/{seq}_{meth}_FULL.mp4"
        if not (os.path.exists(man_p) and os.path.exists(vid_p)):
            print("missing", seq, meth); continue
        man, v = json.load(open(man_p)), gl.read_video(vid_p)
        vids[meth] = v
        for b in man["boundaries"]:          # 4 frames around every seam: b-2, b-1 | b, b+1
            rows.append(np.concatenate([label(small(v[j], 3), f"{meth} seam@{b} f{j}") for j in (b - 2, b - 1, b, b + 1)], axis=1))
    if rows:
        Image.fromarray(np.concatenate(rows, axis=0)).save(f"{OUT}/figures/{seq}_seams.jpg", quality=88)
    if vids:                                  # side-by-side video, shorter ones hold their last frame
        L = max(v.shape[0] for v in vids.values())
        grid = [np.concatenate([label(small(v[min(t, v.shape[0] - 1)]), k) for k, v in vids.items()], axis=1) for t in range(L)]
        export_to_video([g.astype(np.float32) / 255.0 for g in grid], f"{OUT}/videos/{seq}_COMPARE.mp4", fps=FPS)
    print(seq, "->", {k: f"{v.shape[0] / FPS:.1f}s" for k, v in vids.items()})

recs = [json.loads(l) for w in WORKERS for l in open(f"{OUT}/logs/runs_worker{w}.jsonl") if l.strip()]
new = [r for r in recs if "sec" in r]
bad = [r["tag"] for r in new if not r.get("frames_ok", True)]
print(f"\n{len(new)} clips generated | mean {np.mean([r['sec'] for r in new]):.0f}s per clip | "
      f"max peak {max(r['peak_gb'] for r in new):.2f} GB | black/NaN clips: {bad or 'none'}")
json.dump(recs, open(f"{OUT}/logs/day2_runs_all.json", "w"), indent=1)
!du -sh {OUT} | cat

## What to send back
- `figures/*_seams.jpg` (4 images) and your impressions of `videos/*_COMPARE.mp4`
- the last printed summary line (clips, seconds per clip, peak GB, black clips)
- if a worker failed: the end of `logs/worker0.log` / `worker1.log`

**Next (Day 3):** metrics notebook (CLIPScore, DINO/CLIP consistency, seam smoothness, VBench-style motion
smoothness) on these videos, and LoRA training for multi-frame continuation.